# Overview

Tokenization comprises several steps:

-   Normalization (any cleanup of the text that is deemed necessary, such as removing spaces or accents, Unicode normalization, etc.)
-   Pre-tokenization (splitting the input into words)
-   Running the input through the model (using the pre-tokenized words to produce a sequence of tokens)
-   Post-processing (adding the special tokens of the tokenizer, generating the attention mask and token type IDs)

More precisely, the library is built around a central `Tokenizer` class with the building blocks regrouped in submodules:

-   `normalizers` contains all the possible types of `Normalizer` you can use (complete list [here](https://huggingface.co/docs/tokenizers/api/normalizers)).
-   `pre_tokenizers` contains all the possible types of `PreTokenizer` you can use (complete list [here](https://huggingface.co/docs/tokenizers/api/pre-tokenizers)).
-   `models` contains the various types of `Model` you can use, like `BPE`, `WordPiece`, and `Unigram` (complete list [here](https://huggingface.co/docs/tokenizers/api/models)).
-   `trainers` contains all the different types of `Trainer` you can use to train your model on a corpus (one per type of model; complete list [here](https://huggingface.co/docs/tokenizers/api/trainers)).
-   `post_processors` contains the various types of `PostProcessor` you can use (complete list [here](https://huggingface.co/docs/tokenizers/api/post-processors)).
-   `decoders` contains the various types of `Decoder` you can use to decode the outputs of tokenization (complete list [here](https://huggingface.co/docs/tokenizers/components#decoders)).

You can find the whole list of building blocks [here](https://huggingface.co/docs/tokenizers/components).

# Acquiring a corpus

In [ ]:
from datasets import load_dataset
dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")
def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i : i + 1000]["text"]

# Building a WordPiece tokenizer from scratch

- we start by instantiating a `Tokenizer` object with a `model`
- then set its `normalizer`, `pre_tokenizer`, `post_processor`, and `decoder` attributes to the values we want.
- We have to specify the `unk_token` so the model knows what to return when it encounters characters it hasn’t seen before.
    - Other arguments we can set here include the `vocab` of our model (we’re going to train the model, so we don’t need to set this)
    - and `max_input_chars_per_word`, which specifies a maximum length for each word (words longer than the value passed will be split).

## Model

In [ ]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

## Normalizer

- There is a `BertNormalizer` with the classic options we can set for BERT:
    - `lowercase`
    - `strip_accents`
    - `clean_text` to remove all control characters and replace repeating spaces with a single one
    - `handle_chinese_chars`, which places spaces around Chinese characters.
- To replicate the `bert-base-uncased` tokenizer, we can just set this normalizer: ```tokenizer.normalizer = normalizers.BertNormalizer(lowercase=True)```
- However, when building a new tokenizer you won’t have access to such a handy normalizer already implemented in the 🤗 Tokenizers library
- so let’s see how to create the BERT normalizer by hand.
    - The library provides a `Lowercase` normalizer and a `StripAccents` normalizer
    - and you can compose several normalizers using a `Sequence`:
    - We’re also using an NFD Unicode normalizer, as otherwise the StripAccents normalizer won’t properly recognize and strip out the accented characters.
---
- We can use the `normalize_str()` method of the `normalizer` to check out the effects it has on a given text.

In [ ]:
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents()]
)

print(tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))

## Pre-Tokenizer

- Again, there is a prebuilt `BertPreTokenizer` that we can use: `tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()`
- Or we can build it from scratch.
    - Note that the `Whitespace` pre-tokenizer splits on whitespace and all characters that are not letters, digits, or the underscore character
    - So, it technically splits on whitespace and punctuation
    - If you only want to split on whitespace, you should use the `WhitespaceSplit` pre-tokenizer instead.
    - Like with normalizers, you can use a `Sequence` to compose several pre-tokenizers.

In [ ]:
test_case = "Let's test my pre-tokenizer."
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
print(tokenizer.pre_tokenizer.pre_tokenize_str(test_case))
print("===================")
tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
print(tokenizer.pre_tokenizer.pre_tokenize_str(test_case))
print("===================")
pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.WhitespaceSplit(), pre_tokenizers.Punctuation()])
print(pre_tokenizer.pre_tokenize_str(test_case))
print("===================")

## Trainer

- As well as specifying the `vocab_size` and `special_tokens`, we can set
- the `min_frequency` (the number of times a token must appear to be included in the vocabulary)
- or change the `continuing_subword_prefix` (if we want to use something different from `##`).

In [ ]:
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
# TODO: how does the trainer know what does each special token mean?
trainer = trainers.WordPieceTrainer(vocab_size=25000, special_tokens=special_tokens)

## Train

### Using generator

In [ ]:
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
test_text = "Let's test this tokenizer."
encoding = tokenizer.encode(test_text)
print(encoding.tokens)

- The `encoding` obtained is an `Encoding`, which contains all the necessary outputs of the tokenizer in its various attributes:
    - `ids`, `type_ids`, `tokens`, `offsets`, `attention_mask`, `special_tokens_mask`, and `overflowing`.

### Using local text file

In [ ]:
# with open("wikitext-2.txt", "w", encoding="utf-8") as f:
#     for i in range(len(dataset)):
#         f.write(dataset[i]["text"] + "\n")
# tokenizer.model = models.WordPiece(unk_token="[UNK]")
# tokenizer.train(["wikitext-2.txt"], trainer=trainer)
# test_text = "Let's test this tokenizer."
# encoding = tokenizer.encode(test_text)
# print(encoding.tokens)

## Post-Processing

- We will use a `TemplateProcessor` for this, but first we need to know the IDs of the `[CLS]` and `[SEP]` tokens in the vocabulary.
- To write the template for the `TemplateProcessor`, we have to specify how to treat a single sentence and a pair of sentences.
- For both, we write the special tokens we want to use
    - the first (or single) sentence is represented by `$A`,
    - while the second sentence (if encoding a pair) is represented by `$B`.
    - For each of these (special tokens and sentences), we also specify the corresponding **token type ID** after a colon.

- The classic **BERT template** is thus defined as follows

In [ ]:
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print(cls_token_id, sep_token_id)
print("=================")

tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)

encoding = tokenizer.encode(test_text)
print(encoding.tokens)
print(encoding.type_ids)
print("=================")
encoding = tokenizer.encode(test_text, test_text)
print(encoding.tokens)
print(encoding.type_ids)
print("=================")

## Decoder

In [ ]:
tokenizer.decoder = decoders.WordPiece(prefix="##")
tokenizer.decode(encoding.ids)

## Save and Load

In [ ]:
tokenizer.save("tokenizer.json")
new_tokenizer = Tokenizer.from_file("tokenizer.json")
tokenizer.decode(encoding.ids)

## Make it fast!!

- To use this tokenizer in 🤗 Transformers, we have to wrap it in a `PreTrainedTokenizerFast`.
    - We can either use the generic class:

        ```from transformers import BertTokenizerFast; wrapped_tokenizer = BertTokenizerFast(tokenizer_object=tokenizer)```
    
    - or, if our tokenizer corresponds to an existing model, use that class (here, `BertTokenizerFast`).

- To wrap the tokenizer in a `PreTrainedTokenizerFast`
    - we can either pass the tokenizer we built as a `tokenizer_object`
    - or pass the tokenizer file we saved as `tokenizer_file`.
- The key thing to remember is that we have to manually set all the special tokens
    - since that class can’t infer from the `tokenizer` object which token is the mask token, the `[CLS]` token, etc.
- You can then use this tokenizer like any other 🤗 Transformers tokenizer.
    - You can save it with the `save_pretrained()` method
    - or upload it to the Hub with the `push_to_hub()` method.

In [ ]:
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    # tokenizer_file="tokenizer.json", # alternatively
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)

tokenizer.decode(encoding.ids)